# 🚀 OmniVoice TTS - Google Colab GPU T4 Worker

Notebook này cho phép bạn tận dụng **GPU NVIDIA Tesla T4 (16GB VRAM) miễn phí** trên Google Colab để tăng tốc sinh âm thanh cho dự án **self-tts**, đặc biệt phù hợp khi chạy máy tính cá nhân **không có card đồ hoạ rời (VGA)**.

### 📌 Hướng dẫn sử dụng (Chỉ 3 bước):
1. **Bật GPU:** Vào menu `Runtime` (Thời gian chạy) ➔ `Change runtime type` (Thay đổi loại thời gian chạy) ➔ Chọn **T4 GPU** ➔ Bấm **Save**.
2. **Chạy ô bên dưới:** Bấm nút **Play (▶️)** ở ô code phía dưới và đợi thông báo sẵn sàng (khoảng 1-2 phút lần đầu).
3. **Kết nối máy tính:** Sao chép đường link `https://xxxx.trycloudflare.com` được in ra và dán vào file `backend/.env` trên máy của bạn:
   ```env
   USE_COLAB_GPU=true
   COLAB_API_URL=https://xxxx.trycloudflare.com
   ```

In [ ]:
#@title ⚡ Khởi Động OmniVoice GPU Worker & Cloudflare Tunnel (Bấm Run)
import os
import sys
import time
import re
import subprocess
import urllib.request
import json

# 1. Kiểm tra GPU CUDA
print("🔍 [1/5] Đang kiểm tra card đồ hoạ GPU...")
gpu_check = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
if gpu_check.returncode != 0:
    raise RuntimeError("❌ Chưa bật GPU! Vui lòng vào menu: Runtime -> Change runtime type -> Chọn T4 GPU rồi bấm chạy lại ô này.")
print("✅ Nhận diện GPU thành công!\n" + gpu_check.stdout.split("\n")[8])

# 2. Cài đặt các thư viện cần thiết
print("\n📦 [2/5] Đang cài đặt thư viện cần thiết (OmniVoice, Accelerate, Transformers, FastAPI, Uvicorn)...")
reqs = [
    "omnivoice", "accelerate>=0.34.0", "webdataset", "safetensors", "transformers>=4.40.0",
    "soundfile", "librosa", "pydub", "scipy", "faster-whisper", "fastapi", "uvicorn[standard]",
    "python-multipart", "httpx"
]
subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + reqs, check=True)
print("✅ Đã cài đặt xong toàn bộ thư viện phụ thuộc!")

# 3. Tạo file colab_worker.py trực tiếp
print("\n⚙️ [3/5] Đang thiết lập mã nguồn OmniVoice Colab Worker...")
worker_code = '''import os
import io
import gc
import re
import tempfile
import logging
from pathlib import Path
import torch
import soundfile as sf
import numpy as np
from fastapi import FastAPI, HTTPException, UploadFile, File, Form
from fastapi.responses import Response
from fastapi.middleware.cors import CORSMiddleware
from omnivoice import OmniVoice, VoiceClonePrompt

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] colab_worker — %(message)s", datefmt="%H:%M:%S")
logger = logging.getLogger("colab_worker")
SAMPLE_RATE = 24_000
MODEL_ID = os.getenv("OMNIVOICE_MODEL_ID", "k2-fsa/OmniVoice")

app = FastAPI(title="OmniVoice Colab GPU Worker")
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_credentials=True, allow_methods=["*"], allow_headers=["*"])
_model = None

def get_gpu_info():
    if not torch.cuda.is_available():
        return {"device": "cpu", "gpu_name": "CPU", "vram_total_gb": 0.0}
    props = torch.cuda.get_device_properties(0)
    return {"device": "cuda:0", "gpu_name": torch.cuda.get_device_name(0), "vram_total_gb": round(props.total_memory / (1024**3), 2)}

def load_worker_model():
    global _model
    if _model is not None:
        return
    device = "cuda:0" if torch.cuda.is_available() else "cpu"
    dtype = torch.float16 if torch.cuda.is_available() else torch.float32
    if torch.cuda.is_available():
        torch.backends.cudnn.benchmark = True
    logger.info(f"🚀 Đang tải mô hình {MODEL_ID} lên {device}...")
    _model = OmniVoice.from_pretrained(MODEL_ID, device_map=device, dtype=dtype)
    logger.info("✅ OmniVoice đã nạp thành công vào GPU T4!")

@app.on_event("startup")
def startup_event():
    load_worker_model()

@app.get("/api/remote/health")
def health_check():
    return {"status": "ok", "model_loaded": _model is not None, **get_gpu_info()}

def clean_vietnamese_text(text):
    if not text: return ""
    text = re.sub(r":\\s*", ", ", text)
    text = re.sub(r";\\s*", ", ", text)
    for q in [chr(34), chr(8220), chr(8221), chr(39), chr(8216), chr(8217), chr(171), chr(187)]: text = text.replace(q, "")
    text = re.sub(r"\\.{2,}", ".", text)
    text = re.sub(r"-{2,}", "-", text)
    return re.sub(r"[ \\t]+", " ", text).strip()

def split_into_chunks(text, max_chars=450):
    cleaned = clean_vietnamese_text(text)
    paragraphs = [p.strip() for p in re.split(r"\\n+", cleaned) if p.strip()]
    chunks = []
    for p in paragraphs:
        if len(p) <= max_chars:
            chunks.append(p)
            continue
        sentences = re.split(r"(?<=[.?!…])\\s+", p)
        cur = ""
        for s in sentences:
            s = s.strip()
            if not s: continue
            if not cur: cur = s
            elif len(cur) + len(s) + 1 <= max_chars: cur += " " + s
            else: chunks.append(cur); cur = s
        if cur: chunks.append(cur)
    final_chunks = []
    for c in chunks:
        c = c.strip()
        if c and not c.endswith((".", "!", "?", "…")): c += "."
        if c: final_chunks.append(c)
    return final_chunks or [cleaned]

@app.post("/api/remote/prompt")
async def create_prompt_endpoint(audio_file: UploadFile = File(...), ref_text: str | None = Form(default=None)):
    if _model is None: raise HTTPException(status_code=503, detail="Mô hình chưa tải")
    suffix = Path(audio_file.filename or "sample.wav").suffix or ".wav"
    with tempfile.NamedTemporaryFile(delete=False, suffix=suffix) as tmp:
        tmp.write(await audio_file.read())
        tmp_path = tmp.name
    try:
        clean_text = ref_text.strip() if (ref_text and ref_text.strip()) else None
        prompt = _model.create_voice_clone_prompt(ref_audio=tmp_path, ref_text=clean_text, preprocess_prompt=True)
        with tempfile.NamedTemporaryFile(delete=False, suffix=".pt") as tmp_pt:
            prompt.save(tmp_pt.name)
            with open(tmp_pt.name, "rb") as pf: content = pf.read()
            os.remove(tmp_pt.name)
        return Response(content=content, media_type="application/octet-stream")
    finally:
        try: os.remove(tmp_path)
        except OSError: pass

@app.post("/api/remote/generate")
async def generate_endpoint(
    text: str = Form(...), mode: str = Form(default="clone"), instruct: str | None = Form(default=None),
    num_step: int = Form(default=16), cfg_value: float = Form(default=2.0), speed: float = Form(default=1.0),
    seed: int | None = Form(default=42), ref_text: str | None = Form(default=None),
    prompt_file: UploadFile | None = File(default=None), ref_audio_file: UploadFile | None = File(default=None),
):
    if _model is None: raise HTTPException(status_code=503, detail="Mô hình chưa tải")
    if seed is not None:
        torch.manual_seed(seed)
        if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
    voice_clone_prompt = None
    tmp_ref_path = None
    try:
        if prompt_file is not None:
            pt_bytes = await prompt_file.read()
            with tempfile.NamedTemporaryFile(delete=False, suffix=".pt") as tmp_pt:
                tmp_pt.write(pt_bytes)
                tmp_pt_path = tmp_pt.name
            try:
                dev_str = "cuda:0" if torch.cuda.is_available() else "cpu"
                voice_clone_prompt = VoiceClonePrompt.load(tmp_pt_path, map_location=dev_str)
            except Exception as e:
                logger.warning(f"Không thể đọc prompt_file: {e}")
            finally:
                try: os.remove(tmp_pt_path)
                except OSError: pass

        if ref_audio_file is not None:
            sfx = Path(ref_audio_file.filename or "ref.wav").suffix or ".wav"
            with tempfile.NamedTemporaryFile(delete=False, suffix=sfx) as tmp:
                tmp.write(await ref_audio_file.read())
                tmp_ref_path = tmp.name

        chunks = split_into_chunks(text, max_chars=450)
        logger.info(f"⚡ [Colab GPU] Xử lý {len(chunks)} chunks | num_step={num_step} | Text: '{text[:40]}…'")
        all_audios = []
        design_voice_clone_prompt = None
        with torch.inference_mode():
            for chunk in chunks:
                gen_kwargs = {"text": chunk, "language": "vi", "num_step": num_step, "guidance_scale": cfg_value, "normalize_text": False, "speed": speed}
                if mode == "clone":
                    if voice_clone_prompt is not None: gen_kwargs["voice_clone_prompt"] = voice_clone_prompt
                    elif tmp_ref_path:
                        gen_kwargs["ref_audio"] = tmp_ref_path
                        if ref_text and ref_text.strip(): gen_kwargs["ref_text"] = clean_vietnamese_text(ref_text)
                elif mode == "design":
                    if design_voice_clone_prompt is not None: gen_kwargs["voice_clone_prompt"] = design_voice_clone_prompt
                    elif instruct: gen_kwargs["instruct"] = instruct
                audio_list = _model.generate(**gen_kwargs)
                if audio_list and len(audio_list) > 0:
                    anp = np.array(audio_list[0], dtype=np.float32)
                    if anp.ndim > 1: anp = anp.squeeze()
                    all_audios.append(anp)
        if not all_audios: raise HTTPException(status_code=500, detail="Không sinh được âm thanh")
        silence_array = np.zeros(int(SAMPLE_RATE * 0.22), dtype=np.float32)
        final_pieces = []
        fade_len = int(SAMPLE_RATE * 0.01)
        for i, a in enumerate(all_audios):
            if len(a) > fade_len * 2:
                a[:fade_len] *= np.linspace(0, 1, fade_len, dtype=np.float32)
                a[-fade_len:] *= np.linspace(1, 0, fade_len, dtype=np.float32)
            final_pieces.append(a)
            if i < len(all_audios) - 1: final_pieces.append(silence_array)
        final_audio = np.concatenate(final_pieces)
        wav_buf = io.BytesIO()
        sf.write(wav_buf, final_audio, SAMPLE_RATE, format="WAV")
        wav_buf.seek(0)
        return Response(content=wav_buf.getvalue(), media_type="audio/wav")
    finally:
        if tmp_ref_path: 
            try: os.remove(tmp_ref_path)
            except OSError: pass
        if torch.cuda.is_available(): torch.cuda.empty_cache()

if __name__ == '__main__':
    import uvicorn
    uvicorn.run(app, host='127.0.0.1', port=8000)
'''
with open("colab_worker.py", "w", encoding="utf-8") as f:
    f.write(worker_code)

# 4. Khởi động server ngầm và ghi log
print("\n🚀 [4/5] Đang khởi động OmniVoice GPU Worker & nạp mô hình vào VRAM...")
log_file = open("worker.log", "w", encoding="utf-8")
server_proc = subprocess.Popen([sys.executable, "-u", "colab_worker.py"], stdout=log_file, stderr=subprocess.STDOUT)

# Chờ server sẵn sàng bằng cách poll health endpoint (tối đa 90s)
is_ready = False
print("⏳ Đang tải trọng số mô hình k2-fsa/OmniVoice (vui lòng đợi ~30-60 giây)...", end="", flush=True)
for _ in range(45):
    if server_proc.poll() is not None:
        # Tiến trình đã kết thúc bất thường
        break
    try:
        with urllib.request.urlopen("http://127.0.0.1:8000/api/remote/health", timeout=2) as resp:
            if resp.status == 200:
                data = json.loads(resp.read().decode())
                is_ready = True
                print(f"\n✅ Server đã sẵn sàng! [{data.get('gpu_name')} - {data.get('vram_total_gb')}GB VRAM]")
                break
    except Exception:
        pass
    print(".", end="", flush=True)
    time.sleep(2)

if not is_ready:
    log_file.close()
    with open("worker.log", "r", encoding="utf-8") as f:
        err_output = f.read()
    raise RuntimeError(f"❌ Server không khởi động được! Chi tiết log lỗi:\n\n{err_output}")

# 5. Tải và kích hoạt Cloudflare Tunnel
print("\n🌐 [5/5] Đang kết nối đường hầm Cloudflare Tunnel...")
subprocess.run(["wget", "-q", "-nc", "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb"], check=True)
subprocess.run(["dpkg", "-i", "-E", "cloudflared-linux-amd64.deb"], capture_output=True)

tunnel_proc = subprocess.Popen(["cloudflared", "tunnel", "--url", "http://127.0.0.1:8000"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

tunnel_url = None
start_time = time.time()
while time.time() - start_time < 30:
    line = tunnel_proc.stdout.readline()
    if not line: continue
    match = re.search(r"https://[-a-zA-Z0-9]+\.trycloudflare\.com", line)
    if match:
        tunnel_url = match.group(0)
        break

if tunnel_url:
    print("\n" + "="*70)
    print("🎉 GOOGLE COLAB GPU (T4 16GB) ĐÃ SẴN SÀNG HOẠT ĐỘNG!")
    print(f"👉 URL kết nối: {tunnel_url}")
    print("="*70)
    print("\n📋 HÃY SAO CHÉP 2 DÒNG DƯỚI ĐÂY DÁN VÀO FILE backend/.env TRÊN MÁY BẠN:")
    print("-"*70)
    print(f"USE_COLAB_GPU=true")
    print(f"COLAB_API_URL={tunnel_url}")
    print("-"*70 + "\n")
    print("💡 Giữ tab Google Colab này mở trong suốt quá trình sử dụng hệ thống.\n")
    tunnel_proc.wait()
else:
    print("⚠️ Không tìm thấy URL Cloudflared trong 30s. Vui lòng thử khởi động lại ô này.")
